Adapted from: https://medium.com/@rajan.mehta911/how-to-plug-any-encoder-into-gliner2-32c590180260

# GLiNER2 Custom Encoder Integration

This notebook demonstrates how to build and test a custom GLiNER2 architecture. We will swap out the default base model (`deberta-v3`) for a custom Hugging Face model (`distilbert-base-uncased`), inspect the tokenized sequences, and run end-to-end inference!

It could be useful if:

- you have an existing domain-specific encoder, and want to share the same weights for NER (or other tasks) rather than maintaining two separate models.
- you want to use a language specific encoder
- you want to swap in a static model (model2vec, BPEmb) and get sub-millisecond encoding
- you need edge-deployment to microcontrollers, mobile apps, or Raspberry Pi that can't load a 400MB BERT.

### Step 1: Install Dependencies
Ensure you have `gliner2` installed.

In [ ]:
!pip install gliner2

### Step 2: Understand How the Encoder Sees Sequences

- Before writing our custom GLiNER2 class, we need to understand how GLiNER2's `SchemaTransformer` builds the input sequence.

- **Key insight:** Special tokens like `[CLS]` and `[SEP]` are **never automatically added**. Instead, GLiNER2 uses its own structure:
```
schema_tokens [SEP_TEXT] text_tokens
```

Below we verify this with `distilbert-base-uncased`.

In [7]:
from transformers import AutoTokenizer
from gliner2.processor import SchemaTransformer

# Set our target model
MODEL_NAME = "P0L3/clirebert_clirevocab_uncased"

# Now: see what SchemaTransformer actually builds
processor = SchemaTransformer(MODEL_NAME)
dummy_data = [("Hello, custom GLiNER model!", {"entities": {"Person": "", "Location": ""}})]
batch = processor.collate_fn_train(dummy_data)

print(processor.tokenizer.decode(batch.input_ids[0]))
print("Schema comes FIRST (( [P] entities ( [E] ... ) )), then [SEP_TEXT], then the text.")

( [P] entities ( [E] person [E] location ) ) [SEP_TEXT] hell ##o , custom gli ##ner model !
Schema comes FIRST (( [P] entities ( [E] ... ) )), then [SEP_TEXT], then the text.


GLiNER2 processes text and schema labels together in a single sequence. Let's look at exactly what `SchemaTransformer` builds so we know how it reverse-indexes it.

In [8]:
print(f"\nbatch.original_lengths[0]: {batch.original_lengths[0]} tokens")
print("This is the TOTAL non-padded sequence length (schema + [SEP_TEXT] + text).")

print(f"\nbatch.mapped_indices[0]: {batch.mapped_indices[0]}")
print("Per-subword tuples: (seg_type, orig_idx, schema_idx)")


batch.original_lengths[0]: 19 tokens
This is the TOTAL non-padded sequence length (schema + [SEP_TEXT] + text).

batch.mapped_indices[0]: [('schema', 0, 0), ('schema', 1, 0), ('schema', 2, 0), ('schema', 3, 0), ('schema', 4, 0), ('schema', 5, 0), ('schema', 6, 0), ('schema', 7, 0), ('schema', 8, 0), ('schema', 9, 0), ('sep', 10, 1), ('text', 11, 1), ('text', 11, 1), ('text', 12, 1), ('text', 13, 1), ('text', 14, 1), ('text', 14, 1), ('text', 15, 1), ('text', 16, 1)]
Per-subword tuples: (seg_type, orig_idx, schema_idx)


### Step 3: Building Custom Extractor

- The entire heavy lifting is done by a single class: the `Extractor`. The core neural network in GLiNER2 is the `Extractor`. It receives a `PreprocessedBatch` (containing `original_lengths` and `mapped_indices` discussed above) and computes the loss or predictions.

- To inject your own encoder model, you create a class that inherits from `GLiNER2` and override its `__init__` and `_encode_batch` methods. That's it.

- Everything before encoding (schema serialization, tokenization) and everything after encoding (splitting, scoring, thresholding) remains unchanged :)

- Entire call chain:
```
extract_entities(text, labels)
    │
    ├── Schema.build()                           ← untouched (Step 1)
    │
    ├── processor.collate_fn_inference()         ← untouched (Step 2)
    │       serializes schema + text into
    │       one flat token sequence
    │
    ├── YOUR_ENCODER(input_ids, attn_mask)       ← THE SWAP (Step 3)
    │       .last_hidden_state                   YOU PROVIDE THIS.
    │
    ├── processor.extract_embeddings_from_batch()  ← untouched (Step 4)
    │       splits hidden states back into
    │       token_embs and schema_embs
    │
    ├── compute_span_rep()                       ← untouched (Step 5)
    ├── count_pred()                             ← untouched (Step 6)
    ├── count_embed() + einsum                   ← untouched (Step 7)
    └── _extract_entities / _extract_structures / _extract_relations
                                                 ← untouched (Step 8)
```

In [9]:
import torch
import torch.nn as nn
from transformers import AutoModel
from gliner2 import GLiNER2
from gliner2.model import ExtractorConfig
from gliner2.processor import PreprocessedBatch
from gliner.modeling.span_rep import SpanRepLayer
from gliner2.layers import CountLSTM, create_mlp


class CustomGLiNER2Config(ExtractorConfig):
    model_type = "custom_extractor"


class CustomExtractor(GLiNER2):
    """
    Custom GLiNER2 model that swaps the default encoder for a different HuggingFace model.

    Inheriting from GLiNER2 (not just Extractor) gives us extract_entities,
    batch_extract, and the complete inference pipeline for free.
    """

    @classmethod
    def _can_set_experts_implementation(cls):
        # This model has no MoE experts, so returning False is both correct and safe.
        return False

    def __init__(self, config: CustomGLiNER2Config, tokenizer=None):
        super().__init__(config, tokenizer=tokenizer)

        # 1. Swap in our custom encoder.
        self.encoder = AutoModel.from_pretrained(config.model_name)

        # 2. IMPORTANT: resize embeddings to include SchemaTransformer's special tokens
        # ([SEP_TEXT], [P], [E], [C], [R], [L], etc. are added to the tokenizer vocab).
        self.encoder.resize_token_embeddings(len(self.processor.tokenizer))

        # 3. Re-initialize projection layers for our encoder's hidden size.
        # Only required if your custom encoder has a DIFFERENT hidden_size than the base model.
        hidden_size = self.encoder.config.hidden_size

        self.span_rep = SpanRepLayer(
            span_mode="markerV0",
            hidden_size=hidden_size,
            max_width=config.max_width,
            dropout=0.1,
        )

        self.classifier = create_mlp(
            input_dim=hidden_size, intermediate_dims=[hidden_size * 2],
            output_dim=1, dropout=0.0, activation="relu", add_layer_norm=False
        )

        self.count_pred = create_mlp(
            input_dim=hidden_size, intermediate_dims=[hidden_size * 2],
            output_dim=20, dropout=0.0, activation="relu", add_layer_norm=False
        )
        self.count_embed = CountLSTM(hidden_size)

    def _encode_batch(self, batch: PreprocessedBatch):
        """
        Training path override.

        NOTE: The inference path (extract_entities → _extract_from_batch) does
        the same thing directly, bypassing this method. Both paths are consistent.
        """
        outputs = self.encoder(
            input_ids=batch.input_ids,
            attention_mask=batch.attention_mask,
        )
        return self.processor.extract_embeddings_from_batch(
            outputs.last_hidden_state, batch.input_ids, batch
        )

### Step 4: End-to-End Inference

- With `CustomExtractor` inheriting from `GLiNER2`, we instantiate it directly
- The `SchemaTransformer` processor is automatically created from `config.model_name`.
- **Note**: Since the projection layers have random weights, predictions will be noisy. This just confirms the forward pass runs end-to-end without errors.

In [10]:
config = CustomGLiNER2Config(model_name=MODEL_NAME)
pipeline = CustomExtractor(config)
pipeline.eval()

Some weights of BertModel were not initialized from the model checkpoint at P0L3/clirebert_clirevocab_uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Some weights of BertModel were not initialized from the model checkpoint at P0L3/clirebert_clirevocab_uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🧠 Model Configuration
Encoder model      : P0L3/clirebert_clirevocab_uncased
Counting layer     : count_lstm
Token pooling      : first


CustomExtractor(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30532, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

In [11]:
sample_text = "Apple CEO Tim Cook announced the new iPhone 15 in Cupertino, California."
labels = ["person", "company", "location", "product"]

print(f"Text: '{sample_text}'")
print("Running extract_entities... (untrained model — predictions are noise)")

# Low threshold since the scorer hasn't been fine-tuned
predictions = pipeline.extract_entities(sample_text, labels, threshold=0.001)
print("\nPredictions:", predictions)

Text: 'Apple CEO Tim Cook announced the new iPhone 15 in Cupertino, California.'
Running extract_entities... (untrained model — predictions are noise)

Predictions: {'entities': {'person': ['Cook announced the new', 'California.', 'Tim', 'CEO', 'iPhone 15 in Cupertino', 'Apple', ','], 'company': ['Tim Cook announced the', 'new iPhone 15 in Cupertino, California', 'CEO', 'Apple', '.'], 'location': ['CEO Tim Cook announced the', 'iPhone 15 in Cupertino, California', 'Apple', 'new', '.'], 'product': ['Tim Cook announced the new iPhone 15 in', 'CEO', 'Apple', 'California', '.', ',', 'Cupertino']}}


### 5: Fine-Tuning Your Distilbert GLiNER

Once you run the trainer on your dataset, the `SpanRepLayer` and `CountLSTM` will learn to map the distinct label vectors (`[SEP_TEXT], [P], [E], [C], [R], [L], etc.`) to their corresponding clusters in the text.

More info on training:
- GLiNER2 dataset format: [training-dataset.md](https://github.com/fastino-ai/GLiNER2/blob/main/tutorial/8-train_data.md)
(I created [this app](https://gliner2-dataset-generator.streamlit.app/) to generate synthetic data for free using ollama cloud)
- GLiNER2 training tutorial: [training.md](https://github.com/fastino-ai/GLiNER2/blob/main/tutorial/9-training.md)


In [ ]:
from gliner2.training.trainer import TrainingConfig, GLiNER2Trainer
from gliner2.training.data import TrainingDataset

train_dataset = TrainingDataset.load("unie_synthetic_ner_gliner2.jsonl")

training_config = TrainingConfig(
    output_dir="model2vec-gliner",
    num_epochs=3,
    batch_size=8,
    encoder_lr=1e-5, # Keep small so we don't destroy original embeddings
    task_lr=5e-4,    # Large LR for our custom span scoring MLPs
    fp16=True if torch.cuda.is_available() else False,
    use_lora=False,
)

trainer = GLiNER2Trainer(
    model=pipeline,
    config=training_config,
)

trainer.train(train_data=train_dataset)

Let's retry the same query again...

In [ ]:
sample_text = "Apple CEO Tim Cook announced the new iPhone 15 in Cupertino, California."
labels = ["person", "company", "location", "product"]

print(f"Text: '{sample_text}'")
print("Running extract_entities... (untrained model — predictions are noise)")

# Low threshold since the scorer hasn't been fine-tuned
predictions = pipeline.extract_entities(sample_text, labels, threshold=0.001)
print("\nPredictions:", predictions)

Text: 'Apple CEO Tim Cook announced the new iPhone 15 in Cupertino, California.'
Running extract_entities... (untrained model — predictions are noise)

Predictions: {'entities': {'person': ['Tim Cook', 'Apple CEO', 'Cupertino', 'iPhone 15'], 'company': ['Cupertino, California', 'Apple CEO', 'Tim Cook', 'iPhone 15'], 'location': ['Cupertino, California', 'Apple', 'Tim Cook', 'iPhone'], 'product': ['iPhone 15', 'Apple CEO']}}


As can be seen, the predictions are slightly better than before!
